[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zjelveh/zjelveh.github.io/blob/master/files/ccjs418e_fall2026/11_strengthening_comparisons.ipynb)

**Save your own copy.**
1. Click File > Save a copy in Drive.
2. Rename it.
3. Work in your copy.

---

# 11. Strengthening Comparisons: Did the Bronx Change More Than Brooklyn?
## CCJS 418E: Coding for Criminology — Fall 2026

Notebook 10 ended with a gap: during the 2023 season, the Bronx averaged 429.0 noise complaints on Yankees home-game days and 400.8 on other days, about 28 more per game day. This notebook asks how much of that gap belongs to the stadium.

> Did Bronx noise complaints rise more on game days than complaints in a borough without a stadium?

The same three questions guide the work:

1. What do you want to know?
2. Compared to what?
3. What can you honestly say?

In [ ]:
# Show the result of every line in a cell, so one cell can display several outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Print NumPy numbers as plain values, such as 3 instead of np.int64(3)
np.set_printoptions(legacy="1.25")
# Draw charts on a white background with light grid lines
sns.set_style('whitegrid')
# Make every chart 9 inches wide and 4 inches tall
plt.rcParams['figure.figsize'] = (9, 4)

## Setup: The Same Two Tables

These are notebook 10's tables. Part 6 adds a third.

**`complaints`: one row represents one 311 noise complaint** in the Bronx or Brooklyn in 2023, with the columns `created_date`, `borough`, `complaint_type`, and `descriptor`.

**`games`: one row represents one Yankees home game** in the 2023 regular season, with the columns `game_date`, `opponent`, and `result`.

In [ ]:
complaints_url = 'https://raw.githubusercontent.com/zjelveh/zjelveh.github.io/refs/heads/master/files/ccjs418e_fall2026/nyc_311_noise_sample.csv'
games_url = 'https://raw.githubusercontent.com/zjelveh/zjelveh.github.io/refs/heads/master/files/ccjs418e_fall2026/yankees_home_games_2023.csv'

complaints = pd.read_csv(filepath_or_buffer=complaints_url)
games = pd.read_csv(filepath_or_buffer=games_url)

## Part 1: The Question and the Plan

**What do you want to know?** Whether Yankees home games add noise complaints in the Bronx.

**Compared to what?** Notebook 10 compared Bronx game days with other Bronx days. Game days also differ from other days in ways that reach the whole city: they fall on warm evenings, weekends, and holidays such as July 4. A borough without a stadium, observed on the same dates, can stand in for what those dates would bring the Bronx without a game. `complaints` already includes one: Brooklyn.

**What can you honestly say?** We will decide at the end, after checking the result two ways.

The plan:

1. Count complaints per day in each borough.
2. Mark the game days and keep the season, as in notebook 10.
3. Compare the Bronx's change on game days with Brooklyn's change on the same dates.
4. Check the result with a complaint type a game cannot affect.
5. Compare the hours before and after 7 p.m. on game nights and other nights.

Last week's reading from The Trace compared D.C. shootings with the same weeks a year earlier. A second city observed over the same weeks would show what happened where no troops were deployed. Brooklyn plays that role here.

## Part 2: Count Complaints per Day in Each Borough

Repeat notebook 10's first steps: convert `created_date` to datetime values, label each complaint with its day, and convert the game dates.

In [ ]:
complaints['created_date'] = pd.to_datetime(arg=complaints['created_date'])
complaints['date'] = complaints['created_date'].dt.to_period(freq='D').dt.to_timestamp()
games['date'] = pd.to_datetime(arg=games['game_date'])

complaints[['created_date', 'date', 'borough']].head(n=3)

Notebook 10 kept the Bronx and grouped by `date`. Here we keep both boroughs and group by two columns, `date` and `borough`, as in the groupby notebook. Each row of `daily` represents one borough on one day.

In [ ]:
daily = complaints.groupby(by=['date', 'borough']).size().reset_index(name='complaints')

daily.head(n=4)
daily.shape

Check the table before using it. 2023 had 365 days, so each borough should appear 365 times, and the daily counts should add up to the number of complaints.

In [ ]:
daily['borough'].value_counts()
daily['complaints'].sum()
len(complaints)

Mark the game days and add the day of the week, as in notebook 10.

`game_day` describes the **date**: a Brooklyn row is marked `True` when the Yankees played at home in the Bronx that day. That is what lets Brooklyn stand in for game dates without a stadium.

In [ ]:
daily['game_day'] = daily['date'].isin(values=games['date'])
daily['weekday'] = daily['date'].dt.day_name()

daily['game_day'].sum()
daily.head(n=4)

`daily` has 160 game-day rows: the 80 game dates, once for each borough.

## Part 3: Does Brooklyn Rise Too?

Start with notebook 10's first benchmark, all other days of 2023, now for both boroughs. Grouping by `borough` and `game_day` gives four averages.

In [ ]:
daily.groupby(by=['borough', 'game_day'])['complaints'].mean().round(decimals=1)

The Bronx rose from 310.8 to 429.0, the gap of 118.2 from notebook 10. Brooklyn, miles from the stadium, rose from 370.0 to 466.0, about 96 complaints, on the same dates. Most of the Bronx's jump shows up in Brooklyn too, which fits notebook 10's finding: game days fall in the warm months, when noise complaints rise everywhere.

Now keep the season, as in notebook 10.

In [ ]:
first_game = games['date'].min()
last_game = games['date'].max()
season = daily[daily['date'].between(left=first_game, right=last_game)]

len(season)
season.groupby(by=['borough', 'game_day'])['complaints'].mean().round(decimals=1)

During the season, the Bronx rose from 400.8 on non-game days to 429.0 on game days, an increase of 28.2. Brooklyn rose from 446.0 to 466.0, an increase of 20.0.

Draw one line per borough, from its non-game average to its game-day average. `game_day` is `False` or `True`, which a chart places at 0 and 1, the way `.sum()` counts each `True` as 1. `plt.xticks()` replaces those two numbers with labels.

In [ ]:
season_avg = season.groupby(by=['borough', 'game_day'])['complaints'].mean().round(decimals=1).reset_index(name='avg_complaints')

sns.lineplot(data=season_avg, x='game_day', y='avg_complaints', hue='borough')
# Label the two ends of the x-axis
plt.xticks(ticks=[0, 1], labels=['Non-game days', 'Game days'])
plt.title(label='Average Noise Complaints per Day, 2023 Season')
# Remove the x-axis title
plt.xlabel(xlabel='')
plt.ylabel(ylabel='Complaints per day')

Brooklyn's line sits higher: Brooklyn has more residents and more complaints on both game days and non-game days. Both lines rise from non-game days to game days. If game days added the same number of complaints in both boroughs, the lines would be parallel. The Bronx line rises a little more steeply, 28.2 against 20.0.

The y-axis starts near 400, which makes both rises look steep. Compare the two slopes with each other.

## Part 4: Difference-in-Differences

Calculate each borough's change from non-game days to game days, then subtract Brooklyn's change from the Bronx's.

1. **Bronx change:** game-day average minus non-game average.
2. **Brooklyn change:** the same calculation for Brooklyn.
3. **Extra Bronx increase:** the Bronx change minus the Brooklyn change.

Because the result is a difference between two differences, this comparison is called a **difference-in-differences**.

In [ ]:
bronx = season[season['borough'] == 'BRONX']
brooklyn = season[season['borough'] == 'BROOKLYN']

bronx_change = bronx[bronx['game_day'] == True]['complaints'].mean() - bronx[bronx['game_day'] == False]['complaints'].mean()
brooklyn_change = brooklyn[brooklyn['game_day'] == True]['complaints'].mean() - brooklyn[brooklyn['game_day'] == False]['complaints'].mean()

bronx_change.round(decimals=1)
brooklyn_change.round(decimals=1)

In [ ]:
(bronx_change - brooklyn_change).round(decimals=1)

The Bronx rose 8.2 complaints per day more than Brooklyn on game days.

Each subtraction removes something:

- **Within each borough**, game days are compared with non-game days in the same borough, so Brooklyn's higher level drops out. Only the changes are compared.
- **Between the boroughs**, Brooklyn's change of 20.0 stands in for everything game dates bring to the whole city: warm evenings, weekends, July 4. Subtracting it leaves the part of the Bronx's increase that Brooklyn did not share. Part 5 asks when that stand-in works.

In notebook 10, the gap shrank from 118.2 to 28.2 as the benchmark improved. Comparing with Brooklyn shrinks it again, to 8.2.

## Part 5: What the Comparison Assumes

The difference-in-differences treats Brooklyn's change on game dates as the change the Bronx would have had on those dates without a game. That holds when whatever else happens on game dates moves the two boroughs by about the same number of complaints. Ask what could move one borough more than the other on those dates: a street festival in only one borough, a storm that hits one harder, or Brooklyn's larger population turning the same warm evening into more extra complaints.

Fans arriving, traffic, vendors, and celebrations after a win are the ways a game would produce noise. They are part of the game's effect, the thing we want to measure. A rival explanation is something that happens on game dates for another reason and affects one borough more than the other, such as the festival or storm above.

## Part 6: A Placebo Outcome

A **placebo outcome** is something the cause being studied should leave unchanged. A baseball game cannot make a boiler fail, so heating complaints should show no game-day effect. If heating complaints show an extra Bronx increase as large, relative to their level, as the noise result, then the noise result is the size this comparison finds even for an outcome the games cannot change.

**`heating`: one row represents one 311 heating complaint** (`HEAT/HOT WATER`) in the Bronx or Brooklyn in 2023, with the columns `created_date`, `borough`, and `complaint_type`.

In [ ]:
heating_url = 'https://raw.githubusercontent.com/zjelveh/zjelveh.github.io/refs/heads/master/files/ccjs418e_fall2026/nyc_311_heating_sample.csv'
heating = pd.read_csv(filepath_or_buffer=heating_url)

heating.shape
heating.head(n=3)

Build the daily table with the same steps as Part 2, then keep the season.

In [ ]:
heating['created_date'] = pd.to_datetime(arg=heating['created_date'])
heating['date'] = heating['created_date'].dt.to_period(freq='D').dt.to_timestamp()

heating_daily = heating.groupby(by=['date', 'borough']).size().reset_index(name='complaints')
heating_daily['game_day'] = heating_daily['date'].isin(values=games['date'])
heating_season = heating_daily[heating_daily['date'].between(left=first_game, right=last_game)]

heating_daily.shape
len(heating_season)

In [ ]:
heating_season.groupby(by=['borough', 'game_day'])['complaints'].mean().round(decimals=1)

Calculate the difference-in-differences with the same steps as Part 4.

In [ ]:
heating_bronx = heating_season[heating_season['borough'] == 'BRONX']
heating_brooklyn = heating_season[heating_season['borough'] == 'BROOKLYN']

heating_bronx_change = heating_bronx[heating_bronx['game_day'] == True]['complaints'].mean() - heating_bronx[heating_bronx['game_day'] == False]['complaints'].mean()
heating_brooklyn_change = heating_brooklyn[heating_brooklyn['game_day'] == True]['complaints'].mean() - heating_brooklyn[heating_brooklyn['game_day'] == False]['complaints'].mean()

heating_bronx_change.round(decimals=1)
heating_brooklyn_change.round(decimals=1)
(heating_bronx_change - heating_brooklyn_change).round(decimals=1)

During the season the Bronx averaged about 65 heating complaints a day and Brooklyn about 50, far fewer than in winter. On game days the Bronx rose by 1.9 and Brooklyn by 0.5, an extra Bronx increase of 1.4. (The rounded averages above suggest 0.6 for Brooklyn; the calculation uses the unrounded averages.)

Read 1.4 against the level of heating complaints: it is about 2% of 65. The noise result, 8.2, is also about 2% of the Bronx's noise level. A complaint type that a baseball game cannot affect shows a Bronx-specific change of the same relative size as noise.

A placebo outcome tests the method itself. Heating complaints, which a game cannot affect, show an extra Bronx increase as large, relative to their level, as noise. The noise result is about the size the comparison finds for an outcome the games leave unchanged; a general rise of about 2% in Bronx 311 calls on game dates would produce both. A placebo can weaken or strengthen a causal reading; it cannot show that the games caused the noise result.

## Part 7: Within the Day — Before and After 7 p.m.

Every comparison so far counted all 24 hours of each day. A night game can add noise only during and after the game, so a sharper comparison looks at the hours around it. Assume each game starts at 7 p.m. (all twelve Friday home games in 2023 were scheduled for 7:05 p.m.; ten of the twelve Wednesday games were, and two were afternoon games).

Start with Bronx Fridays during the season: 12 with a home game and 14 without. For each group, compare the four hours before 7 p.m. (3:00–6:59) with the four hours after (7:00–10:59):

| | 3–7 p.m. | 7–11 p.m. |
|---|---|---|
| Game Fridays | before the game | during and after the game |
| Other Fridays | the same hours | the same hours |

Complaints rise every evening, game or no game. The difference-in-differences asks whether they rise **more** on game Fridays: the change on game Fridays minus the change on other Fridays. Here the comparison group is other Fridays in the same borough, and "before" and "after" are hours of the day.

Each complaint needs its hour, its weekday, and whether its date had a game. `.dt.hour` gives the hour of a datetime value as a number from 0 to 23, so 19 means 7:00–7:59 p.m.

In [ ]:
complaints['hour'] = complaints['created_date'].dt.hour
complaints['weekday'] = complaints['date'].dt.day_name()
complaints['game_day'] = complaints['date'].isin(values=games['date'])

complaints[['created_date', 'hour', 'weekday', 'game_day']].head(n=3)

Keep the Bronx complaints made on Fridays during the season, and count the Fridays in each group.

In [ ]:
fridays = complaints[(complaints['borough'] == 'BRONX') & (complaints['weekday'] == 'Friday') & (complaints['date'].between(left=first_game, right=last_game))]

fridays.groupby(by=['game_day'])['date'].nunique()

Count the complaints in each hour for each group, then divide by the number of Fridays in the group. The result, `per_day`, is the average number of complaints in that hour on one Friday. The merge adds each group's number of Fridays to its rows, as the merging notebook added each county's population.

In [ ]:
friday_hours = fridays.groupby(by=['game_day', 'hour']).size().reset_index(name='complaints')
friday_days = fridays.groupby(by=['game_day'])['date'].nunique().reset_index(name='days')
friday_hours = pd.merge(left=friday_hours, right=friday_days, on='game_day')
friday_hours['per_day'] = friday_hours['complaints'] / friday_hours['days']

friday_hours.head(n=3)
friday_hours.shape

In [ ]:
sns.lineplot(data=friday_hours, x='hour', y='per_day', hue='game_day')
plt.axvline(x=19, color='gray', linestyle='--')
plt.title(label='Bronx Noise Complaints by Hour, Fridays in the 2023 Season')
plt.xlabel(xlabel='Hour of the day (19 = 7 p.m.)')
plt.ylabel(ylabel='Complaints per Friday')

The two lines follow each other through the morning and afternoon. After 7 p.m. (the dashed line), the game-Friday line climbs above the other, with the largest gaps from 9 p.m. to midnight, when games end and fans head home.

Add up the four hours before 7 p.m. and the four hours after, for each group. `.between()` keeps the hours from `left` through `right`, including both.

In [ ]:
before = friday_hours[friday_hours['hour'].between(left=15, right=18)]
after = friday_hours[friday_hours['hour'].between(left=19, right=22)]

before.groupby(by=['game_day'])['per_day'].sum().round(decimals=1)
after.groupby(by=['game_day'])['per_day'].sum().round(decimals=1)

In [ ]:
game_change = after[after['game_day'] == True]['per_day'].sum() - before[before['game_day'] == True]['per_day'].sum()
non_game_change = after[after['game_day'] == False]['per_day'].sum() - before[before['game_day'] == False]['per_day'].sum()

game_change.round(decimals=1)
non_game_change.round(decimals=1)
(game_change - non_game_change).round(decimals=1)

From 3–7 p.m. to 7–11 p.m., complaints rose by 129.9 on game Fridays and by 93.4 on other Fridays. The extra increase on game Fridays is 36.5 complaints per evening, the largest game-specific number in this notebook, and it falls in the hours when a game is happening and ending.

Now repeat every step for Wednesdays: the same code, with `'Wednesday'` in place of `'Friday'`.

In [ ]:
wednesdays = complaints[(complaints['borough'] == 'BRONX') & (complaints['weekday'] == 'Wednesday') & (complaints['date'].between(left=first_game, right=last_game))]

wednesday_hours = wednesdays.groupby(by=['game_day', 'hour']).size().reset_index(name='complaints')
wednesday_days = wednesdays.groupby(by=['game_day'])['date'].nunique().reset_index(name='days')
wednesday_hours = pd.merge(left=wednesday_hours, right=wednesday_days, on='game_day')
wednesday_hours['per_day'] = wednesday_hours['complaints'] / wednesday_hours['days']

wednesday_days

sns.lineplot(data=wednesday_hours, x='hour', y='per_day', hue='game_day')
plt.axvline(x=19, color='gray', linestyle='--')
plt.title(label='Bronx Noise Complaints by Hour, Wednesdays in the 2023 Season')
plt.xlabel(xlabel='Hour of the day (19 = 7 p.m.)')
plt.ylabel(ylabel='Complaints per Wednesday')

In [ ]:
before = wednesday_hours[wednesday_hours['hour'].between(left=15, right=18)]
after = wednesday_hours[wednesday_hours['hour'].between(left=19, right=22)]

game_change = after[after['game_day'] == True]['per_day'].sum() - before[before['game_day'] == True]['per_day'].sum()
non_game_change = after[after['game_day'] == False]['per_day'].sum() - before[before['game_day'] == False]['per_day'].sum()

game_change.round(decimals=1)
non_game_change.round(decimals=1)
(game_change - non_game_change).round(decimals=1)

Wednesdays point the other way. In the chart, the game-Wednesday line stays below the other from 7 p.m. to midnight. From 3–7 p.m. to 7–11 p.m., complaints rose by 41.8 on game Wednesdays and by 68.9 on other Wednesdays, an extra increase of −27.1: game Wednesday evenings rose *less*.

Each of these numbers rests on about a dozen evenings per group, and one unusual evening moves it a lot. On April 12, a Wednesday without a game, Bronx complaints rose by 202 from 3–7 p.m. to 7–11 p.m., about three times the typical Wednesday rise. Without that one evening, the Wednesday result is −16.0. Two weekdays, measured the same way, give opposite answers of similar size.

## Part 8: Answer the Question

| Comparison | Bronx change | Brooklyn change | Extra Bronx increase |
|---|---:|---:|---:|
| Game days vs. non-game days in the season | +28.2 | +20.0 | +8.2 |
| Heating complaints in the season (placebo) | +1.9 | +0.5 | +1.4 |

| Bronx evenings, 3–7 p.m. to 7–11 p.m. | Game nights | Other nights | Extra increase on game nights |
|---|---:|---:|---:|
| Fridays | +129.9 | +93.4 | +36.5 |
| Wednesdays | +41.8 | +68.9 | −27.1 |

**What do we want to know?** Whether Yankees home games add noise complaints in the Bronx.

**Compared to what?** For whole days, the Bronx's change on game days compared with Brooklyn's change on the same dates. For evenings, the change from before 7 p.m. to after on game nights compared with the same change on other nights of the same weekday.

**What can we honestly say?** During the 2023 season, the Bronx averaged about 28 more noise complaints per day on Yankees home-game days than on other days, and Brooklyn, with no stadium, averaged about 20 more on the same dates. The extra Bronx increase is about 8 complaints per day, about 2% of the Bronx's level, and heating complaints, which a game cannot affect, show an extra Bronx increase of the same relative size. At the level of daily borough counts, these data show no clear increase specific to the stadium. Looking at the hours around 7 p.m. gives a sharper comparison: on Friday game nights, Bronx complaints rose 36.5 more from late afternoon to evening than on other Fridays, while on Wednesday game nights they rose 27.1 less than on other Wednesdays. With about a dozen game nights per weekday, and opposite answers on the two weekdays, these data leave the stadium's effect on evening noise unsettled. Friday nights are where to look next.

The gap began at 118 against all other days of 2023 and fell to 28 within the season. Comparing with Brooklyn brought it to 8. Hours sharpened the comparison and showed how much a dozen evenings can vary.

**What would strengthen the answer?** More seasons would add many more game nights to each weekday. The stadium also sits in one corner of a borough of 1.4 million people; a table with each complaint's location would allow comparing the blocks around the stadium with the rest of the Bronx.

---

## Hands-On Exercises

Each exercise has two parts and ends with a short written answer. Exercise 1 practices the comparisons in this notebook. Exercise 2 goes further, using the weekday-matched benchmark from notebook 10.

### Exercise 1: Wins and Losses in Brooklyn

Notebook 10 compared Bronx complaints after wins and after losses. Brooklyn has no stadium, so Brooklyn's own gap between win days and loss days shows how much two groups of dates differ for reasons unrelated to the game.

**Part A:** Merge `games` with `brooklyn`, the Brooklyn season days from Part 4, on `date`, keeping every game. Average `complaints` by `result`.

**Part B:** Repeat Part A with `bronx`. Notebook 10's Part 8 found Bronx loss games 30.9 complaints above their weekday-matched benchmark and win games 9.7 above. Answer in 2–3 sentences: what does Brooklyn's gap between win days and loss days tell you about reading that difference?

In [ ]:
# Part A: Merge games with brooklyn and average complaints by result.

In [ ]:
# Part B: Repeat Part A with bronx.

**Your answer:**

### Exercise 2 (Going Further): Match by Weekday

Notebook 10's Part 7 compared each Bronx game day with non-game days on the same weekday and found a gap of 20.2. Do the same for both boroughs.

**Part A:** Using `brooklyn`, average the non-game days by `weekday`, look up each game day's weekday average with a merge, and display the game-day average and the benchmark.

**Part B:** Repeat Part A with `bronx`, then subtract Brooklyn's gap from the Bronx's. Answer in 2–3 sentences: how does this weekday-matched extra Bronx increase compare with Part 4's 8.2?

In [ ]:
# Part A: Build the weekday-matched benchmark for brooklyn.

In [ ]:
# Part B: Build it for bronx and subtract Brooklyn's gap from the Bronx's.

**Your answer:**

## Reference: Comparing Changes in Two Groups

```python
# Count records per day for each group
daily = complaints.groupby(by=['date', 'borough']).size().reset_index(name='complaints')

# Four averages: each group on each kind of day
season.groupby(by=['borough', 'game_day'])['complaints'].mean()

# Difference-in-differences: the change in one group minus the change in the other
bronx_change = bronx[bronx['game_day'] == True]['complaints'].mean() - bronx[bronx['game_day'] == False]['complaints'].mean()
brooklyn_change = brooklyn[brooklyn['game_day'] == True]['complaints'].mean() - brooklyn[brooklyn['game_day'] == False]['complaints'].mean()
bronx_change - brooklyn_change

# Placebo outcome: the same four averages for heating complaints
heating_season.groupby(by=['borough', 'game_day'])['complaints'].mean()

# Hours: the hour of each complaint, then complaints per hour per day for each group
complaints['hour'] = complaints['created_date'].dt.hour
friday_hours = fridays.groupby(by=['game_day', 'hour']).size().reset_index(name='complaints')
friday_days = fridays.groupby(by=['game_day'])['date'].nunique().reset_index(name='days')
friday_hours = pd.merge(left=friday_hours, right=friday_days, on='game_day')
friday_hours['per_day'] = friday_hours['complaints'] / friday_hours['days']
```

| Check | What it asks | Result here |
|---|---|---|
| Second group | Did the comparison group change too? | Brooklyn rose 20.0 vs. the Bronx's 28.2 |
| Placebo outcome | Does an outcome the cause cannot affect show a gap? | Heating: +1.4, about 2% of its level |
| Hours of the day | Did the hours after the game rise more than on other nights? | Fridays +36.5, Wednesdays −27.1 |